In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

#Desactivar los warnings
import warnings
warnings.filterwarnings("ignore")

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

print("✅ Librerías importadas correctamente")

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Mis Casos(Importante)\PROYECTO 5 - FORECASTING RETAIL\02_FORECASTING
✅ Librerías importadas correctamente


In [3]:
x = pd.read_pickle("../02_datos/03_Entrenamiento/x_preseleccionado.pkl")
y = pd.read_pickle("../02_datos/03_Entrenamiento/y_preseleccionado.pkl")

## EMPAQUETAR TODO EL PROCESO EN FUNCIONES

### CALIDAD DE DATOS

In [7]:
#Variables madre segun el Excel de procesos: solo estas entran al proceso
VAR_MADRE = ['store_id', 'item_id', 'event_name_1', 'event_type_1', 'month', 'sell_price', 'ventas', 'weekday']

def calidad_datos(x):
    temp = x[VAR_MADRE].copy()
    #modificar tipos: solo month, porque month_2 es variable final y el OHE se entreno con month en texto
    #(year y wday no llegan a ninguna variable final, convertirlas no sirve para nada)
    temp['month'] = temp['month'].astype('str')
    #imputar nulos de eventos, event_type_1_Sin_evento es variable final
    temp.fillna({'event_name_1': 'Sin_evento', 'event_type_1': 'Sin_evento'}, inplace=True)
    #eliminar registros sin precio (pre lanzamiento, como los 71 de FOODS_3_288), sin reset_index para no perder date
    temp = temp.dropna(subset=['sell_price'])
    return temp

### CREACION DE VARIABLES

In [5]:
def crear_variables(x):
    #demanda intermitente
    def rotura_stock(ventas, n):
        a = (ventas==0).astype(int)
        b = a.rolling(n).sum()
        return(b==n).astype(int)

    #Por si acaso se ordena el dataframe antes de agrupar
    x = x.sort_values(['store_id', 'item_id', 'date'])
    nivelPxT = x.groupby(['store_id', 'item_id'])['ventas']

    for n in (3, 7, 15):
        x[f'rotura_stock_{n}'] = nivelPxT.transform(rotura_stock, n=n)

    # ── Lags ─────────────────────────────────────────────────────────────
    # 'columna_original': [lags en dias] -> para añadir/quitar lags solo se toca esto
    CONFIG_LAGS = {
        'ventas':          list(range(1, 16)),
        'sell_price':      list(range(1, 8)),
        'rotura_stock_3':  [1],
        'rotura_stock_7':  [1],
        'rotura_stock_15': [1],
    }

    #Creacion de lags
    def crear_lags(df, config, cols_serie=('store_id', 'item_id'), col_fecha='date'):
        """Lags dentro de cada serie producto-tienda. Devuelve un df de lags (sin unir)."""
        df = df.sort_values(list(cols_serie) + [col_fecha])
        grupos = df.groupby(list(cols_serie))
        resultado = pd.DataFrame(index=df.index)

        for columna in config:
            for lag in config[columna]:
                resultado[f'{columna}_lag{lag}'] = grupos[columna].shift(lag)

        return resultado

    df_lags = crear_lags(x, CONFIG_LAGS)

    #Ventanas moviles
    #Estadisticos min y max y media movil ponderada
    def estadistico_movil_serie(ventas, n, estadistico):
        return ventas.shift(1).rolling(n).agg(estadistico)

    def media_movil_ponderada_serie(ventas, n):
        return ventas.shift(1).ewm(span=n, min_periods=n).mean()

    for n in range(2, 16):
        x['ventasmin' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='min')
        x['ventasmax' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='max')
        x['ventasmmp' + str(n)] = nivelPxT.transform(media_movil_ponderada_serie, n=n)

    #unir todos los lags con el dataframe original
    x = pd.concat([x, df_lags], axis=1)
    #Eliminar las FILAS que quedaron con nulos por los lags (los primeros 15 dias de cada serie)
    #con subset para no depender de otras columnas con nulos como event_name_2
    x = x.dropna(subset=df_lags.columns)
    #Variables que ya no se van a necesitar (las originales)
    x = x.drop(columns=
               ['dia_ind','wm_yr_wk','sell_price','rotura_stock_3','rotura_stock_7','rotura_stock_15'])
    # Se crea una sola variable para el producto debido a que se haran 20 modelos, una por cada
    # producto-tienda, se eliminan entonces las columnas 'item_id' y 'store_id'
    # mismo orden que en la modelizacion (tienda_producto) para que coincida con lista_modelos
    x['producto'] = x['store_id'] + '_' + x['item_id']
    x = x.drop(columns=['item_id', 'store_id'])
    return x

### TRANSFORMACION DE VARIABLES

In [6]:
import joblib

# Las madres de las dummies finales (Excel de procesos)
VAR_OHE = ['event_name_1', 'event_type_1', 'month', 'weekday']
RUTA_OHE = MODELOS / 'ohe_retail.pkl'


def aplicar_ohe(x, ohe):
    # Cambia las categoricas por sus dummies; index=x.index para que cada dummy caiga en su fila
    dummies = pd.DataFrame(ohe.transform(x[VAR_OHE]), columns=ohe.get_feature_names_out(), index=x.index)
    return pd.concat([x.drop(columns=VAR_OHE), dummies], axis=1)


def transformar_variables_entrenamiento(x):
    # Entrenar: el encoder aprende las categorias de TODO el historico y se guarda
    ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore').fit(x[VAR_OHE])
    joblib.dump(ohe, RUTA_OHE)
    return aplicar_ohe(x, ohe)


def transformar_variables_ejecucion(x):
    # Ejecutar: se usa el encoder guardado y no uno nuevo, para que salgan las mismas columnas del entrenamiento
    ohe = joblib.load(RUTA_OHE)
    return aplicar_ohe(x, ohe)

### PRESELECCION DE VARIABLES

In [8]:
def preseleccion_variables(x, y, n_variables=75):
    #Solo en entrenamiento: se queda con las n variables con mas mutual information contra las ventas
    x = x.drop(columns=['producto'], errors='ignore')
    #un valor de MI por columna, con el nombre de la variable como indice
    mi = pd.Series(mutual_info_regression(x, y), index=x.columns)
    #las n mas altas
    entran = mi.nlargest(n_variables).index
    return x[entran]

### MODELIZACION

#### ENTRENAMIENTO

In [9]:
def modelizar(x_producto, y):
    '''
    Esta función es la que hace la modelización individual.

    Recibe los datos de las x y la y de un producto.

    Encuentra los parámetros óptimos para ese producto.

    Devuelve el mejor modelo.
    '''
    #Excluye el producto como variable de modelización
    #y preselecciona con MI las variables de ESTE producto (cada serie elige las suyas)
    x_mod = preseleccion_variables(x_producto, y)

    #Define la validación cruzada
    time_cv = TimeSeriesSplit(5, test_size = 8)

    #Define la parrilla de algoritmos
    pipe = Pipeline([('algoritmo',HistGradientBoostingRegressor(random_state = 42))])

    grid = [
         {'algoritmo': [HistGradientBoostingRegressor(random_state = 42)]
#          'algoritmo__learning_rate': [0.01,0.025,0.05,0.1],
#          'algoritmo__max_iter': [50,100,200],
#          'algoritmo__max_depth': [5,10,20,50],
#          'algoritmo__scoring': ['neg_mean_absolute_error'],   #OJO: no descomentar, el scoring va en el buscador, no en el algoritmo
#          'algoritmo__l2_regularization': [0,0.25,0.5,0.75,1]
         }

    ]

    #Crea los modelos
    random_search = RandomizedSearchCV(estimator = pipe,
                                   param_distributions = grid,
                                   n_iter = 1,
                                   cv = time_cv,
                                   scoring = 'neg_mean_absolute_error',
                                   verbose = 0,
                                   n_jobs = -1,
                                   random_state = 42)   #para que el sorteo de combinaciones se pueda repetir

    random_search.fit(x_mod, y)

    #Reentrena el mejor sobre todos los datos
    #(no hace falta otro .fit: RandomizedSearchCV ya lo hace solo con refit=True, que viene por defecto)
    modelo_final = random_search.best_estimator_

    #Devuelve como salida el mejor modelo
    return modelo_final

In [ ]:
#Donde se guarda la lista con los 20 modelos entrenados
RUTA_MODELOS = MODELOS / 'lista_modelos_retail.pkl'


def lanzar_entrenamiento(df):

    '''
    Esta función va recorriendo todos los productos y llamando a modelizar() para crear una lista total con todos los modelos de todos los productos.

    Recibe el dataframe que sale de crear_variables(): limpio, con la columna producto y con la target (ventas) dentro.

    No devuelve nada, si no que guarda en disco el objeto ya entrenado con todos los modelos.
    '''

    #Transforma UNA sola vez con todo el historico: el encoder aprende de las 20 series juntas y se guarda una vez
    df = transformar_variables_entrenamiento(df)

    lista_productos = list(df['producto'].unique())

    lista_modelos = []

    for cada in lista_productos:

        #Renombra por claridad
        producto = cada
        target = 'ventas'

        #x e y de este producto, sacadas de sus propias filas (la fecha se repite entre series, isin no sirve)
        x = df.loc[df['producto'] == producto].drop(columns=target).copy()
        y = df.loc[df['producto'] == producto, target].copy()

        #Llama a la funcion de modelizar (dentro ya hace la preseleccion con MI de este producto)
        modelo = modelizar(x, y)

        #Añade el modelo final a la lista
        lista_modelos.append((producto, modelo))

    #Guarda la lista de modelos entrenados
    joblib.dump(lista_modelos, RUTA_MODELOS)

#### EJECUCION

In [ ]:
#Lanzar la predicción
def lanzar_ejecucion(df):

    '''
    Esta función hace el forecast para cada producto, pero solo de un día.

    Recibe el dataframe que sale de crear_variables() con los datos nuevos a predecir.

    Va recorriendo cada producto, cargando su modelo correspondiente, seleccionando sus datos, y haciendo las predicciones.

    Devuelve la predicción para todos los productos pero SOLO PARA EL DÍA QUE TOCA.
    '''

    #CARGA LOS MODELOS (la lista [(producto, modelo), ...] que guardo lanzar_entrenamiento)
    lista_modelos = joblib.load(RUTA_MODELOS)

    #Transforma UNA sola vez con el encoder guardado (igual que en el entrenamiento)
    df = transformar_variables_ejecucion(df)

    lista_predicciones = []

    for cada in lista_modelos:

        #Renombra por claridad
        producto = cada[0]
        modelo = cada[1]
        #las variables con las que se entreno ESTE modelo (las que eligio su preseleccion con MI)
        variables = modelo.feature_names_in_
        target = 'ventas'

        #x e y de este producto, sacadas de sus propias filas
        x = df.loc[df.producto == producto].drop(columns=target).copy()
        y = df.loc[df.producto == producto, target].copy()

        #Seleccion de variables: solo las de su modelo y en el mismo orden
        x = x[variables]

        #Cálculo de predicciones (date ya viene en el indice, no hace falta sacarla aparte)
        predicciones = pd.DataFrame({'producto': producto,
                                     'ventas': y,
                                     'prediccion': modelo.predict(x).astype('int')},
                                    index = x.index)

        lista_predicciones.append(predicciones)

    predicciones_df = pd.concat(lista_predicciones)

    #Se queda con el dia que toca: la fecha mas baja que sobrevivio al dropna de los lags
    predicciones_df = predicciones_df.loc[predicciones_df.index == predicciones_df.index.min()]

    #date pasa a columna para el CSV que va a Power BI
    return predicciones_df.reset_index()

In [ ]:
#Dias a predecir
HORIZONTE = 8


def forecast_recursivo(x):

    '''
    Esta función es la que aplica el forecast recursivo para predecir 8 días.

    Recibe el nuevo dataset a predecir (15 dias de historia + los 8 dias a predecir).

    Ya que para aplicar la recursividad:

    * Va a predecir el primer día para el cual tenga toda la información (es decir 15 días desde el día más antiguo)
    * Al finalizar graba la predicción de ventas en el registro a predecir y elimina los registros del día más antiguo del dataframe
    * Por tanto en la siguiente iteración va a predecir el siguiente día.

    Por ejemplo:

    Si el día más antiguo del dataset es el 09/12/2015 entonces el primer día que puede predecir
    (y del cual ya no tenemos dato) es el 24/12/2015.

    Cuando predice el dato del 24 para cada producto lo sobrescribe como sus ventas
    y elimina todos los registros del día 09.

    Entonces el día más antiguo pasa a ser el día 10 y por tanto el día a predecir es el 25.

    Y así hasta que finaliza 8 ciclos para predecir la semana que queremos.
    '''

    for cada in range(0, HORIZONTE):
        paso1_df = calidad_datos(x)
        paso2_df = crear_variables(paso1_df)

        #Calcula la predicción (una fila por producto, solo del dia que toca)
        f = lanzar_ejecucion(paso2_df)

        #Actualiza el dato de ventas con la predicción
        #cada fila de ese dia busca la prediccion de SU producto (tienda_producto), asi no depende del orden de las filas
        prediccion_por_producto = f.set_index('producto')['prediccion']
        dia = f['date'].iloc[0]
        es_dia = x.index == dia
        producto_fila = x.loc[es_dia, 'store_id'] + '_' + x.loc[es_dia, 'item_id']
        x.loc[es_dia, 'ventas'] = producto_fila.map(prediccion_por_producto).values

        #Elimina el día más antiguo de x
        x = x.loc[x.index != x.index.min()]

    return x

## PROCESO

### REENTRANAMIENTO

### EVALUACION

### EJECUCION